In [1]:
import numpy as np
import os
import pandas as pd
import pickle
from tqdm.auto import tqdm

In [2]:
results_dir = '/wynton/home/capra/gramey02/dnd_project/results'
run_name = 'RUN_MULTIALLELIC'
strats=['indels','CRISPRoff','acceptor_base_edits', 'donor_base_edits','excision']
ot_excavate_dir='excavate/excavate_outputs_w_offtargets'
w_excavate_dir='excavate/excavate_outputs_w_wobble'

In [3]:
# run loop across strats and genes
for strat in tqdm(strats, desc='strategies'):
    # make new directory for the results
    os.makedirs(os.path.join(results_dir, run_name, strat, 'excavate', 'excavate_wobble_offtarget_merged'), exist_ok=True)
    if strat!='excision':
        # get the current strats' gene outputs for both wobble annotations and offtarget annotations
        ot_dir = os.path.join(results_dir, run_name, strat, ot_excavate_dir)
        ot_outputs=os.listdir(ot_dir)
        ot_genes={x.split('_')[0] for x in ot_outputs}
    else:
        ot_dir=os.path.join(results_dir, run_name, strat, 'excision_selected_snps_w_offtargets', ot_excavate_dir)
        ot_outputs=os.listdir(ot_dir)
        ot_genes={x.split('_')[0] for x in ot_outputs}

    # get the wobble files
    w_dir=os.path.join(results_dir, run_name, strat, w_excavate_dir)
    w_outputs=os.listdir(w_dir)
    w_genes={x.split('_')[0] for x in w_outputs}

    # assert that the same genes are in each set
    if strat == 'excision':
        if not ot_genes <= w_genes:
            raise ValueError(f'Excision off-target genes missing from wobble run: {ot_genes - w_genes}, when they should be a subset of wobble.')
        # genes that failed pair selection have no off-target run; skip or carry them with NaN off-targets
        missing = w_genes - ot_genes
        if len(missing)>0:
            print(f'Skipped genes not in wobble annotated genes: {missing}.')
    elif ot_genes!=w_genes:
        raise ValueError(f'Genes for {strat} are not equal across excavate runs. Nonoverlapping genes: {ot_genes.symmetric_difference(w_genes)}')

    for gene in tqdm(ot_genes):
        # load the wobble file
        cur_w = pd.read_csv(os.path.join(w_dir, gene+'_output', 'all_guides_w_wobble.csv'))
        # load the off targets file
        cur_ot = pd.read_csv(os.path.join(ot_dir, gene+'_output', 'all_guides.csv'))
        
        # assert that all of the snps in one file are there in the other
        if strat == 'excision':
            if len(cur_ot) > len(cur_w):
                raise ValueError(
                    f'{strat}/{gene}: off-target run has more rows ({len(cur_ot)}) '
                    f'than wobble run ({len(cur_w)})'
                )
        else:
            if len(cur_w) != len(cur_ot):
                raise ValueError(
                    f'{strat}/{gene}: off-target run has {len(cur_ot)} rows, '
                    f'wobble run has {len(cur_w)}'
                )
                
        # merge the two files:
        merged=cur_w.merge(cur_ot, on=['chrom', 'start', 'end', 'rsID', 'SNP position', 'gRNA', 'PAM',
                                       'strand', 'variation', 'present allele', 'guide ID',
                                       'SNP position in protospacer or PAM', 'In population',
                                       'alt allele frequency'],how='left')
        merged.rename(columns={'1 bp mismatches in ch1':'1 bp mismatches in the chromosome'},inplace=True)

        # if strat=excision, fill the Nan offtarget info with an annotation saying that it didn't run because of downsampling during offtarget runs
        # keep numerics numeric, flag in a separate column if off targets were not estimated
        merged['off target estimated because snp in top 50 most frequent'] = merged['exact matches in genome'].notna()
        
        # save in new section of the results
        os.makedirs(os.path.join(results_dir, run_name, strat, 'excavate', 'excavate_wobble_offtarget_merged',gene+'_output'), exist_ok=True)
        merged.to_csv(os.path.join(results_dir, run_name, strat, 'excavate', 'excavate_wobble_offtarget_merged',gene+'_output', 'all_guides.csv'), index=False)

strategies:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/218 [00:00<?, ?it/s]

  0%|          | 0/240 [00:00<?, ?it/s]

  0%|          | 0/136 [00:00<?, ?it/s]

  0%|          | 0/65 [00:00<?, ?it/s]

Skipped genes not in wobble annotated genes: {'H1-4', 'ZIC1'}.


  0%|          | 0/535 [00:00<?, ?it/s]

In [4]:
merged

,chrom,start,end,rsID,SNP position,gRNA,PAM,strand,variation,present allele,guide ID,SNP position in protospacer or PAM,In population,alt allele frequency,targeted allele (genomic),mismatch (non-target allele),CFD in seed (PAM-proximal 10bp),exact matches in genome,1 bp mismatches in the chromosome,off target estimated because snp in top 50 most frequent
0,chr16,71725463,71725482,16:71725461:C:T,71725461,AAAGTAGGTCAGAAAGGGGT,CGG,-,C>T,C (ref),chr16_71725463_-_C_SpCas9_20nt,-2,Y,0.421725,C,rG:dT,0.883,NaN,NaN,False
1,chr16,71725638,71725657,16:71725659:A:G,71725659,TTCCACCTTGAAATTATTTT,CGG,+,A>G,G (alt),chr16_71725638_+_G_SpCas9_20nt,-2,Y,0.425719,G,rG:dT,0.883,0.0,0.0,True
2,chr16,71725644,71725663,16:71725659:A:G,71725659,CTTGAAATTATTTTCGGGAA,AGG,+,A>G,G (alt),chr16_71725644_+_G_SpCas9_20nt,5,Y,0.425719,G,rG:dT,0.883,0.0,1.0,True
3,chr16,71725644,71725663,16:71725659:A:G,71725659,CTTGAAATTATTTTCAGGAA,AGG,+,A>G,A (ref),chr16_71725644_+_A_SpCas9_20nt,5,Y,0.425719,A,rA:dC,0.609,1.0,0.0,True
4,chr16,71725759,71725778,16:71725769:A:G,71725769,GACAGAAGCAAATTTTAACA,AGG,+,A>G,A (ref),chr16_71725759_+_A_SpCas9_20nt,10,Y,0.166733,A,rA:dC,0.609,1.0,0.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
213,chr16,71806010,71806029,16:71806012:T:C,71806012,tatttttattgtttgagAca,ggg,-,T>C,T (ref),chr16_71806010_-_T_SpCas9_20nt,3,Y,0.418730,T,rA:dC,0.609,NaN,NaN,False
214,chr16,71806010,71806029,16:71806012:T:C,71806012,tatttttattgtttgagGca,ggg,-,T>C,C (alt),chr16_71806010_-_C_SpCas9_20nt,3,Y,0.418730,C,rG:dT,0.883,NaN,NaN,False
215,chr16,71806011,71806030,16:71806012:T:C,71806012,ttatttttattgtttgagGc,agg,-,T>C,C (alt),chr16_71806011_-_C_SpCas9_20nt,2,Y,0.418730,C,rG:dT,0.883,NaN,NaN,False
216,chr16,71806011,71806030,16:71806012:T:C,71806012,ttatttttattgtttgagAc,agg,-,T>C,T (ref),chr16_71806011_-_T_SpCas9_20nt,2,Y,0.418730,T,rA:dC,0.609,NaN,NaN,False


## Finally, merge all of the guide information into a supplementary file

In [5]:
import time
merged_dirname = os.path.join('excavate', 'excavate_wobble_offtarget_merged')

def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)
class step:
    """Prints on entry and exit with elapsed time, so a long stage is never silent."""
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.time()
        log(f'-> {self.name} ...')
        return self

    def __exit__(self, *exc):
        log(f'   {self.name} finished in {time.time() - self.t0:.1f}s')

# acceptor and donor are two flavors of the same edit; they collapse to one label
# but strategy_detail below keeps the original distinction available
STRAT_DIRS = ['indels', 'CRISPRoff', 'acceptor_base_edits', 'donor_base_edits', 'excision']
STRAT_LABEL = {
    'indels': 'exon disruption',
    'CRISPRoff': 'epigenetic silencing',
    'acceptor_base_edits': 'splice site disruption',
    'donor_base_edits': 'splice site disruption',
    'excision': 'excision',
}

# derived from the map, never restated - this is what broke last time
LABELS = list(dict.fromkeys(STRAT_LABEL.values()))


def slug(label):
    return label.replace(' ', '_')


FLAG_COLS = [f'targetable_by_{slug(l)}' for l in LABELS]
DETAIL_COLS = ['base_edit_acceptor', 'base_edit_donor']

# what makes a guide row unique
KEY = ['gene', 'chrom', 'start', 'end', 'rsID', 'SNP position', 'gRNA', 'PAM',
       'strand', 'variation', 'present allele', 'guide ID']

# annotations that should agree wherever the same guide appears
STATIC_COLS = ['SNP position in protospacer or PAM', 'In population', 'alt allele frequency',
               'targeted allele (genomic)', 'mismatch (non-target allele)',
               'CFD in seed (PAM-proximal 10bp)']

# these do NOT always agree: NaN for excision SNPs dropped during downsampling
OT_COLS = ['exact matches in genome', '1 bp mismatches in the chromosome']
OT_FLAG = 'off target estimated because snp in top 50 most frequent'

log(f'labels:       {LABELS}')
log(f'flag columns: {FLAG_COLS}')

# ----------------------------------------------------------- load into long form
frames = []
with step('loading per-gene CSVs'):
    for strat_dir in STRAT_DIRS:
        base = os.path.join(results_dir, run_name, strat_dir, merged_dirname)
        entries = sorted(os.listdir(base))
        n_rows = n_empty = 0
        for entry in tqdm(entries, desc=strat_dir, leave=False):
            gene = entry.replace('_output', '')
            df = pd.read_csv(os.path.join(base, entry, 'all_guides.csv'))
            if df.empty:          # genes with no guides; keeps dtypes clean
                n_empty += 1
                continue
            df['gene'] = gene
            df['strategy'] = STRAT_LABEL[strat_dir]
            df['strategy_detail'] = strat_dir
            frames.append(df)
            n_rows += len(df)
        log(f'   {strat_dir}: {len(entries)} genes, {n_rows:,} rows, {n_empty} empty')

with step('concat + dtype normalization'):
    long_df = pd.concat(frames, ignore_index=True)
    del frames
    # concat across files can float-ify these; pin them so groupby keys line up
    for col in ['start', 'end', 'SNP position']:
        long_df[col] = pd.to_numeric(long_df[col], errors='raise').astype('Int64')
    long_df[OT_FLAG] = long_df[OT_FLAG].fillna(False).astype(bool)
    mem = long_df.memory_usage(deep=True).sum() / 1e9
    log(f'   long_df: {long_df.shape[0]:,} rows x {long_df.shape[1]} cols, {mem:.2f} GB')

# -------------------------------------------------------------- sanity guard
# reindex(fill_value=False) below will silently invent all-False columns if a
# label is missing, so fail loudly here instead
seen = set(long_df['strategy'].unique())
missing = [l for l in LABELS if l not in seen]
extra = [s for s in seen if s not in LABELS]
assert not missing and not extra, (
    f'LABELS out of sync with long_df["strategy"] - '
    f'missing from data: {missing}, unexpected in data: {extra}'
)
log(f'   label check passed: {sorted(seen)}')

# ------------------------------------------------- diagnostic: real conflicts?
with step('checking annotation conflicts'):
    grp = long_df.groupby(KEY, dropna=False, sort=False)      # built once, reused
    log(f'   unique guide groups: {grp.ngroups:,}')
    conflicts = grp[STATIC_COLS].nunique(dropna=False)
    conflicts = conflicts[(conflicts > 1).any(axis=1)]
    log(f'   conflicting WOBBLE annotations: {len(conflicts)}  (expect 0)')
    ot_conflicts = grp[OT_COLS].nunique(dropna=True)
    ot_conflicts = int((ot_conflicts > 1).any(axis=1).sum())
    log(f'   guides with 2+ DIFFERING non-null off-target values: {ot_conflicts}')
    log('   (>0 means something beyond a downsampling NaN gap - worth a look)')

# ------------------------------------------- collapse to one row per guide
with step('collapsing to one row per guide'):
    # 'first' already skips NaN, so it prefers the strategy where the
    # off-target value was actually estimated
    agg = {c: 'first' for c in STATIC_COLS + OT_COLS}
    agg[OT_FLAG] = 'max'      # estimated under ANY strategy counts as estimated
    guides = long_df.groupby(KEY, dropna=False, sort=False).agg(agg).reset_index()
    log(f'   guides: {len(guides):,} rows')

with step('building strategy flags'):
    # dedupe first, then unstack: much lighter than pivot_table over every row
    pairs = long_df[KEY + ['strategy']].drop_duplicates()
    flags = (pairs.assign(_v=True)
             .set_index(KEY + ['strategy'])['_v']
             .unstack('strategy', fill_value=False)
             .reindex(columns=LABELS, fill_value=False)
             .rename(columns={l: f'targetable_by_{slug(l)}' for l in LABELS})
             .reset_index())
    log(f'   flags: {flags.shape}')

    # keep the acceptor/donor split even though they share a label
    dpairs = long_df[KEY + ['strategy_detail']].drop_duplicates()
    detail = (dpairs.assign(_v=True)
              .set_index(KEY + ['strategy_detail'])['_v']
              .unstack('strategy_detail', fill_value=False)
              .reindex(columns=['acceptor_base_edits', 'donor_base_edits'], fill_value=False)
              .rename(columns={'acceptor_base_edits': 'base_edit_acceptor',
                               'donor_base_edits': 'base_edit_donor'})
              .reset_index())
    log(f'   detail: {detail.shape}')

with step('merging flags onto guides'):
    guides = guides.merge(flags, on=KEY, how='left').merge(detail, on=KEY, how='left')
    guides[FLAG_COLS] = guides[FLAG_COLS].fillna(False).astype(bool)
    guides[DETAIL_COLS] = guides[DETAIL_COLS].fillna(False).astype(bool)
    # vectorized instead of .apply(axis=1), which built a Series per row
    arr = np.array(LABELS, dtype=object)
    mat = guides[FLAG_COLS].to_numpy()
    guides['n_strategies'] = mat.sum(axis=1)
    guides['strategies'] = [';'.join(arr[r]) for r in mat]
    log(f'   guides: {guides.shape}')

# ------------------------------------------------ SNP-level strategy annotation
SNP_KEY = ['gene', 'chrom', 'rsID', 'SNP position', 'variation']
with step('building SNP-level table'):
    spairs = long_df[SNP_KEY + ['strategy']].drop_duplicates()
    snp_flags = (spairs.assign(_v=True)
                 .set_index(SNP_KEY + ['strategy'])['_v']
                 .unstack('strategy', fill_value=False)
                 .reindex(columns=LABELS, fill_value=False)
                 .rename(columns={l: f'targetable_by_{slug(l)}' for l in LABELS})
                 .reset_index())
    snp_stats = (long_df.groupby(SNP_KEY, dropna=False, sort=False)
                 .agg(n_guides=('guide ID', 'nunique'),
                      alt_allele_frequency=('alt allele frequency', 'first'),
                      min_exact_matches=('exact matches in genome', 'min'),
                      any_off_target_estimated=(OT_FLAG, 'max'))
                 .reset_index())
    snps = snp_flags.merge(snp_stats, on=SNP_KEY, how='left')
    snps[FLAG_COLS] = snps[FLAG_COLS].fillna(False).astype(bool)
    smat = snps[FLAG_COLS].to_numpy()
    snps['n_strategies'] = smat.sum(axis=1)
    snps['strategies'] = [';'.join(arr[r]) for r in smat]
    log(f'   snps: {snps.shape}')

log('ALL DONE')
print(f'\nlong rows:     {len(long_df):,}')
print(f'unique guides: {len(guides):,}')
print(f'unique SNPs:   {len(snps):,}')
print()
print(snps['strategies'].value_counts().head(15))


[13:27:43] labels:       ['exon disruption', 'epigenetic silencing', 'splice site disruption', 'excision']
[13:27:43] flag columns: ['targetable_by_exon_disruption', 'targetable_by_epigenetic_silencing', 'targetable_by_splice_site_disruption', 'targetable_by_excision']
[13:27:43] -> loading per-gene CSVs ...


indels:   0%|          | 0/218 [00:00<?, ?it/s]

[13:27:46]    indels: 218 genes, 1,708 rows, 20 empty


CRISPRoff:   0%|          | 0/240 [00:00<?, ?it/s]

[13:27:49]    CRISPRoff: 240 genes, 2,707 rows, 18 empty


...

[13:27:51]    acceptor_base_edits: 136 genes, 863 rows, 22 empty


donor_base_edits:   0%|          | 0/65 [00:00<?, ?it/s]

[13:27:52]    donor_base_edits: 65 genes, 257 rows, 13 empty


excision:   0%|          | 0/535 [00:00<?, ?it/s]

[13:28:01]    excision: 535 genes, 355,905 rows, 0 empty
[13:28:01]    loading per-gene CSVs finished in 18.3s
[13:28:01] -> concat + dtype normalization ...
[13:28:02]    long_df: 361,440 rows x 23 cols, 0.31 GB
[13:28:02]    concat + dtype normalization finished in 1.2s
[13:28:02]    label check passed: ['epigenetic silencing', 'excision', 'exon disruption', 'splice site disruption']
[13:28:02] -> checking annotation conflicts ...
[13:28:03]    unique guide groups: 356,423
[13:28:03]    conflicting WOBBLE annotations: 0  (expect 0)
[13:28:03]    guides with 2+ DIFFERING non-null off-target values: 0
[13:28:03]    (>0 means something beyond a downsampling NaN gap - worth a look)
[13:28:03]    checking annotation conflicts finished in 0.7s
[13:28:03] -> collapsing to one row per guide ...
[13:28:04]    guides: 356,423 rows
[13:28:04]    collapsing to one row per guide finished in 0.8s
[13:28:04] -> building strategy flags ...
[13:28:06]    flags: (356423, 16)
[13:28:09]    detail: (356

In [14]:
snps

,gene,chrom,rsID,SNP position,variation,targetable_by_exon_disruption,targetable_by_epigenetic_silencing,targetable_by_splice_site_disruption,targetable_by_excision,n_guides,alt_allele_frequency,min_exact_matches,any_off_target_estimated,n_strategies,strategies
0,AARS1,chr16,16:70251998:A:G,70251998,A>G,False,False,False,True,8,0.544329,0.0,True,1,excision
1,AARS1,chr16,16:70253274:A:G,70253274,A>G,False,False,True,True,5,0.880990,0.0,True,2,splice site disruption;excision
2,AARS1,chr16,16:70254829:T:C,70254829,T>C,False,False,False,True,1,0.105831,0.0,True,1,excision
3,AARS1,chr16,16:70258841:T:C,70258841,T>C,False,False,False,True,1,0.880791,0.0,True,1,excision
4,AARS1,chr16,16:70261362:C:G,70261362,C>G,False,False,False,True,2,0.108826,NaN,False,1,excision
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95224,ZSWIM6,chr5,5:61592483:G:T,61592483,G>T,False,False,False,True,1,0.843251,NaN,False,1,excision
95225,ZSWIM6,chr5,5:61592844:A:G,61592844,A>G,False,False,False,True,2,0.101637,NaN,False,1,excision
95226,ZSWIM6,chr5,5:61593466:A:G,61593466,A>G,False,False,False,True,4,0.100040,NaN,False,1,excision
95227,ZSWIM6,chr5,5:61594218:T:G,61594218,T>G,False,False,False,True,3,0.101637,NaN,False,1,excision


In [15]:
snps.strategies.unique()

array(['excision', 'splice site disruption;excision',
       'exon disruption;excision', 'epigenetic silencing;excision',
       'exon disruption;splice site disruption;excision',
       'exon disruption;epigenetic silencing;excision',
       'splice site disruption',
       'exon disruption;epigenetic silencing;splice site disruption;excision',
       'exon disruption;splice site disruption', 'epigenetic silencing',
       'epigenetic silencing;splice site disruption', 'exon disruption',
       'exon disruption;epigenetic silencing',
       'epigenetic silencing;splice site disruption;excision'],
      dtype=object)

In [6]:
guides

,gene,chrom,start,end,rsID,SNP position,gRNA,PAM,strand,variation,...,1 bp mismatches in the chromosome,off target estimated because snp in top 50 most frequent,targetable_by_exon_disruption,targetable_by_epigenetic_silencing,targetable_by_splice_site_disruption,targetable_by_excision,base_edit_acceptor,base_edit_donor,n_strategies,strategies
0,AARS1,chr16,70269658,70269677,16:70269677:G:A,70269677,CCACAGTGATGGTCCGAGCG,TGG,+,G>A,...,0.0,True,True,False,False,True,False,False,2,exon disruption;excision
1,AARS1,chr16,70269658,70269677,16:70269677:G:A,70269677,CCACAGTGATGGTCCGAGCA,TGG,+,G>A,...,1.0,True,True,False,False,True,False,False,2,exon disruption;excision
2,AARS1,chr16,70269674,70269693,16:70269677:G:A,70269677,GGGTGCTGGCTGACCACGCT,CGG,-,G>A,...,0.0,True,True,False,False,True,False,False,2,exon disruption;excision
3,AARS1,chr16,70269674,70269693,16:70269677:G:A,70269677,GGGTGCTGGCTGACCATGCT,CGG,-,G>A,...,1.0,True,True,False,False,True,False,False,2,exon disruption;excision
4,ABCC8,chr11,17396915,17396934,11:17396930:C:A,17396930,CCTTCTGTCCAGGGGCGATG,AGG,+,C>A,...,0.0,True,True,False,True,True,False,True,3,exon disruption;splice site disruption;excision
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356418,ZSWIM6,chr5,61591458,61591477,5:61591475:A:G,61591475,AGAGTGCAGGCATATAGATC,AGG,+,A>G,...,NaN,False,False,False,False,True,False,False,1,excision
356419,ZSWIM6,chr5,61593452,61593471,5:61593466:A:G,61593466,tattcaggcactcaGctgat,tgg,+,A>G,...,NaN,False,False,False,False,True,False,False,1,excision
356420,ZSWIM6,chr5,61593452,61593471,5:61593466:A:G,61593466,tattcaggcactcaActgat,tgg,+,A>G,...,NaN,False,False,False,False,True,False,False,1,excision
356421,ZSWIM6,chr5,61593455,61593474,5:61593466:A:G,61593466,tcaggcactcaActgattgg,tgg,+,A>G,...,NaN,False,False,False,False,True,False,False,1,excision


In [7]:
guides.columns

Index(['gene', 'chrom', 'start', 'end', 'rsID', 'SNP position', 'gRNA', 'PAM',
       'strand', 'variation', 'present allele', 'guide ID',
       'SNP position in protospacer or PAM', 'In population',
       'alt allele frequency', 'targeted allele (genomic)',
       'mismatch (non-target allele)', 'CFD in seed (PAM-proximal 10bp)',
       'exact matches in genome', '1 bp mismatches in the chromosome',
       'off target estimated because snp in top 50 most frequent',
       'targetable_by_exon_disruption', 'targetable_by_epigenetic_silencing',
       'targetable_by_splice_site_disruption', 'targetable_by_excision',
       'base_edit_acceptor', 'base_edit_donor', 'n_strategies', 'strategies'],
      dtype='object')

In [10]:
# save this as a summary file
os.makedirs(os.path.join(results_dir, run_name, 'summary_files','excavate_outputs'), exist_ok=True)
guides.to_csv(os.path.join(results_dir, run_name, 'summary_files','excavate_outputs','gRNA_sequences_w_offtargets.csv'))

## Let's also save the final bed file with all common variants to the summary files location

In [20]:
bed = pd.read_csv('/wynton/home/capra/gramey02/dnd_project/results/RUN_MULTIALLELIC/summary_files/common_var_bed/all_genes.bed', sep='\t', header=None)
bed

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,chr16,70251997,70251998,"rs4985407 A/G, MAF=0.46",0,.,70251997,70251998,"211,211,211",A therapeutically editable common variant for ...,...,"Allele frequency = 0.538767, Heterozygous (AG)...","Allele frequency = 0.443804, Heterozygous (AG)...","Allele frequency = 0.425595, Heterozygous (AG)...","Allele frequency = 0.344581, Heterozygous (AG)...",excision,"70254829,70258841,70260571,70261362,70262711",Yes,ACGCGCGGCTGCGCATGGGCTGTCG_A_GAGAGCGGCGGGAGGCAC...,"CRISPOR - https://crispor.gi.ucsc.edu/, CRISPi...",Please see the browser track main page or go t...
1,chr16,70253273,70253274,"rs4081753 A/G, MAF=0.12",0,.,70253273,70253274,"0,0,0",A therapeutically editable common variant for ...,...,"Allele frequency = 0.797217, Heterozygous (AG)...","Allele frequency = 0.779539, Heterozygous (AG)...","Allele frequency = 0.96627, Heterozygous (AG) ...","Allele frequency = 0.804703, Heterozygous (AG)...","excision,splice site disruption","70254829,70258841,70260571,70261362,70262711",Yes,GGTGCGAGGTGGTGCTGACCTGGGG_A_ACTTGACACAGGCACGTG...,"CRISPOR - https://crispor.gi.ucsc.edu/, CRISPi...",Please see the browser track main page or go t...
2,chr16,70254828,70254829,"rs7205918 T/C, MAF=0.11",0,.,70254828,70254829,"211,211,211",A therapeutically editable common variant for ...,...,"Allele frequency = 0.00397614, Heterozygous (T...","Allele frequency = 0.0317003, Heterozygous (TC...","Allele frequency = 0, Heterozygous (TC) 0% (0/...","Allele frequency = 0, Heterozygous (TC) 0% (0/...",excision,"70251998,70253274,70258841,70260571,70261362",Yes,GCTGCGGAAGCCCCGGCGGTAGGCC_T_TGCAGCAACATACCCCAA...,"CRISPOR - https://crispor.gi.ucsc.edu/, CRISPi...",Please see the browser track main page or go t...
3,chr16,70258840,70258841,"rs7186104 T/C, MAF=0.12",0,.,70258840,70258841,"211,211,211",A therapeutically editable common variant for ...,...,"Allele frequency = 0.797217, Heterozygous (TC)...","Allele frequency = 0.779539, Heterozygous (TC)...","Allele frequency = 0.96627, Heterozygous (TC) ...","Allele frequency = 0.803681, Heterozygous (TC)...",excision,"70251998,70253274,70254829,70260571,70261362",Yes,aagtgctgggattacaggcgtgagc_t_accgtgcccagccTGCTT...,"CRISPOR - https://crispor.gi.ucsc.edu/, CRISPi...",Please see the browser track main page or go t...
4,chr16,70260570,70260571,"rs74024183 T/C, MAF=0.1",0,.,70260570,70260571,"211,211,211",A therapeutically editable common variant for ...,...,"Allele frequency = 0.00298211, Heterozygous (T...","Allele frequency = 0.0288184, Heterozygous (TC...","Allele frequency = 0, Heterozygous (TC) 0% (0/...","Allele frequency = 0, Heterozygous (TC) 0% (0/...",excision,"70251998,70253274,70254829,70258841,70261362",No,GTGAAATCACTGACTTGGGAACATA_T_AGATTAACTCTTAGTTGC...,"CRISPOR - https://crispor.gi.ucsc.edu/, CRISPi...",Please see the browser track main page or go t...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135544,chr5,61592482,61592483,"rs253185 G/T, MAF=0.16",0,.,61592482,61592483,"211,211,211",A therapeutically editable common variant for ...,...,"Allele frequency = 0.981113, Heterozygous (GT)...","Allele frequency = 0.818444, Heterozygous (GT)...","Allele frequency = 0.501984, Heterozygous (GT)...","Allele frequency = 0.861963, Heterozygous (GT)...",excision,"61285544,61286213,61286320,61290062,61290798",Yes,GGTCCCTCTCCTGGCTATCTTGTTT_G_GCTGTAATTATCTCCCAT...,"CRISPOR - https://crispor.gi.ucsc.edu/, CRISPi...",Please see the browser track main page or go t...
135545,chr5,61592843,61592844,"rs16892657 A/G, MAF=0.1",0,.,61592843,61592844,"211,211,211",A therapeutically editable common variant for ...,...,"Allele frequency = 0.00198807, Heterozygous (A...","Allele frequency = 0.0244957, Heterozygous (AG...","Allele frequency = 0, Heterozygous (AG) 0% (0/...","Allele frequency = 0.0245399, Heterozygous (AG...",excision,"61285544,61286213,61286320,61290062,61290798",Yes,CTGAAAAACTGTCAGGGGGCACTTC_A_TGTTGGTGCCGTGGTCAA...,

In [21]:
bed = pd.read_csv('/wynton/home/capra/gramey02/dnd_project/results/RUN_MULTIALLELIC/summary_files/common_var_bed/all_genes.bed', sep='\t', header=None, 
                  names=["chrom", "chromStart", "chromEnd", "rsID", "score", "strand","thickStart", "thickEnd",
                         "itemRgb","Description","Ref/Alt allele","allele_frequency", "minor_allele_frequency", 
                         "Thousand_genomes_genotypes_unrelated_inds",
                         "AFR_genotypes_unrelated_inds",
                         "EUR_genotypes_unrelated_inds",
                         "AMR_genotypes_unrelated_inds",
                         "EAS_genotypes_unrelated_inds",
                         "SAS_genotypes_unrelated_inds",
                         "CRISPR_editing_strategy_targetable_by",
                         "Five closest excision partners, if relevant",
                         "SpCas9 Targetable",
                         "25 bp (+/-) flanking the variant",
                         "Cas/gRNA generation tools",
                         "colorScheme"
    ])
bed

,chrom,chromStart,chromEnd,rsID,score,strand,thickStart,thickEnd,itemRgb,Description,...,EUR_genotypes_unrelated_inds,AMR_genotypes_unrelated_inds,EAS_genotypes_unrelated_inds,SAS_genotypes_unrelated_inds,CRISPR_editing_strategy_targetable_by,"Five closest excision partners, if relevant",SpCas9 Targetable,25 bp (+/-) flanking the variant,Cas/gRNA generation tools,colorScheme
0,chr16,70251997,70251998,"rs4985407 A/G, MAF=0.46",0,.,70251997,70251998,"211,211,211",A therapeutically editable common variant for ...,...,"Allele frequency = 0.538767, Heterozygous (AG)...","Allele frequency = 0.443804, Heterozygous (AG)...","Allele frequency = 0.425595, Heterozygous (AG)...","Allele frequency = 0.344581, Heterozygous (AG)...",excision,"70254829,70258841,70260571,70261362,70262711",Yes,ACGCGCGGCTGCGCATGGGCTGTCG_A_GAGAGCGGCGGGAGGCAC...,"CRISPOR - https://crispor.gi.ucsc.edu/, CRISPi...",Please see the browser track main page or go t...
1,chr16,70253273,70253274,"rs4081753 A/G, MAF=0.12",0,.,70253273,70253274,"0,0,0",A therapeutically editable common variant for ...,...,"Allele frequency = 0.797217, Heterozygous (AG)...","Allele frequency = 0.779539, Heterozygous (AG)...","Allele frequency = 0.96627, Heterozygous (AG) ...","Allele frequency = 0.804703, Heterozygous (AG)...","excision,splice site disruption","70254829,70258841,70260571,70261362,70262711",Yes,GGTGCGAGGTGGTGCTGACCTGGGG_A_ACTTGACACAGGCACGTG...,"CRISPOR - https://crispor.gi.ucsc.edu/, CRISPi...",Please see the browser track main page or go t...
2,chr16,70254828,70254829,"rs7205918 T/C, MAF=0.11",0,.,70254828,70254829,"211,211,211",A therapeutically editable common variant for ...,...,"Allele frequency = 0.00397614, Heterozygous (T...","Allele frequency = 0.0317003, Heterozygous (TC...","Allele frequency = 0, Heterozygous (TC) 0% (0/...","Allele frequency = 0, Heterozygous (TC) 0% (0/...",excision,"70251998,70253274,70258841,70260571,70261362",Yes,GCTGCGGAAGCCCCGGCGGTAGGCC_T_TGCAGCAACATACCCCAA...,"CRISPOR - https://crispor.gi.ucsc.edu/, CRISPi...",Please see the browser track main page or go t...
3,chr16,70258840,70258841,"rs7186104 T/C, MAF=0.12",0,.,70258840,70258841,"211,211,211",A therapeutically editable common variant for ...,...,"Allele frequency = 0.797217, Heterozygous (TC)...","Allele frequency = 0.779539, Heterozygous (TC)...","Allele frequency = 0.96627, Heterozygous (TC) ...","Allele frequency = 0.803681, Heterozygous (TC)...",excision,"70251998,70253274,70254829,70260571,70261362",Yes,aagtgctgggattacaggcgtgagc_t_accgtgcccagccTGCTT...,"CRISPOR - https://crispor.gi.ucsc.edu/, CRISPi...",Please see the browser track main page or go t...
4,chr16,70260570,70260571,"rs74024183 T/C, MAF=0.1",0,.,70260570,70260571,"211,211,211",A therapeutically editable common variant for ...,...,"Allele frequency = 0.00298211, Heterozygous (T...","Allele frequency = 0.0288184, Heterozygous (TC...","Allele frequency = 0, Heterozygous (TC) 0% (0/...","Allele frequency = 0, Heterozygous (TC) 0% (0/...",excision,"70251998,70253274,70254829,70258841,70261362",No,GTGAAATCACTGACTTGGGAACATA_T_AGATTAACTCTTAGTTGC...,"CRISPOR - https://crispor.gi.ucsc.edu/, CRISPi...",Please see the browser track main page or go t...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135544,chr5,61592482,61592483,"rs253185 G/T, MAF=0.16",0,.,61592482,61592483,"211,211,211",A therapeutically editable common variant for ...,...,"Allele frequency = 0.981113, Heterozygous (GT)...","Allele frequency = 0.818444, Heterozygous (GT)...","Allele frequency = 0.501984, Heterozygous (GT)...","Allele frequency = 0.861963, Heterozygous (GT)...",excision,"61285544,61286213,61286320,61290062,61290798",Yes,GGTCCCTCTCCTGGCTATCTTGTTT_G_GCTGTAATTATCTCCCAT...,"CRISPOR - https://crispor.gi.ucsc.edu/, CRISPi...",Please see the browser track main page or go t...
135545,chr5,61592843,61592844,"rs16892657 A/G, MAF=0.1",0,.,61592843,61592844,"211,211,211",A therapeutically editable common variant for ...

In [22]:
# remove some unnecessary columns
bed.drop(labels=['score', 'strand', 'thickStart', 'thickEnd', 'itemRgb', 'Five closest excision partners, if relevant',
                "Cas/gRNA generation tools","colorScheme"], axis=1,inplace=True)
bed

,chrom,chromStart,chromEnd,rsID,Description,Ref/Alt allele,allele_frequency,minor_allele_frequency,Thousand_genomes_genotypes_unrelated_inds,AFR_genotypes_unrelated_inds,EUR_genotypes_unrelated_inds,AMR_genotypes_unrelated_inds,EAS_genotypes_unrelated_inds,SAS_genotypes_unrelated_inds,CRISPR_editing_strategy_targetable_by,SpCas9 Targetable,25 bp (+/-) flanking the variant
0,chr16,70251997,70251998,"rs4985407 A/G, MAF=0.46",A therapeutically editable common variant for ...,A/G,0.54,0.46,"Heterozygous (AG) 41% (1046/2504), Homozygous ...","Allele frequency = 0.839637, Heterozygous (AG)...","Allele frequency = 0.538767, Heterozygous (AG)...","Allele frequency = 0.443804, Heterozygous (AG)...","Allele frequency = 0.425595, Heterozygous (AG)...","Allele frequency = 0.344581, Heterozygous (AG)...",excision,Yes,ACGCGCGGCTGCGCATGGGCTGTCG_A_GAGAGCGGCGGGAGGCAC...
1,chr16,70253273,70253274,"rs4081753 A/G, MAF=0.12",A therapeutically editable common variant for ...,A/G,0.88,0.12,"Heterozygous (AG) 19% (486/2504), Homozygous r...","Allele frequency = 0.98941, Heterozygous (AG) ...","Allele frequency = 0.797217, Heterozygous (AG)...","Allele frequency = 0.779539, Heterozygous (AG)...","Allele frequency = 0.96627, Heterozygous (AG) ...","Allele frequency = 0.804703, Heterozygous (AG)...","excision,splice site disruption",Yes,GGTGCGAGGTGGTGCTGACCTGGGG_A_ACTTGACACAGGCACGTG...
2,chr16,70254828,70254829,"rs7205918 T/C, MAF=0.11",A therapeutically editable common variant for ...,T/C,0.11,0.11,"Heterozygous (TC) 13% (334/2504), Homozygous r...","Allele frequency = 0.381241, Heterozygous (TC)...","Allele frequency = 0.00397614, Heterozygous (T...","Allele frequency = 0.0317003, Heterozygous (TC...","Allele frequency = 0, Heterozygous (TC) 0% (0/...","Allele frequency = 0, Heterozygous (TC) 0% (0/...",excision,Yes,GCTGCGGAAGCCCCGGCGGTAGGCC_T_TGCAGCAACATACCCCAA...
3,chr16,70258840,70258841,"rs7186104 T/C, MAF=0.12",A therapeutically editable common variant for ...,T/C,0.88,0.12,"Heterozygous (TC) 19% (487/2504), Homozygous r...","Allele frequency = 0.98941, Heterozygous (TC) ...","Allele frequency = 0.797217, Heterozygous (TC)...","Allele frequency = 0.779539, Heterozygous (TC)...","Allele frequency = 0.96627, Heterozygous (TC) ...","Allele frequency = 0.803681, Heterozygous (TC)...",excision,Yes,aagtgctgggattacaggcgtgagc_t_accgtgcccagccTGCTT...
4,chr16,70260570,70260571,"rs74024183 T/C, MAF=0.1",A therapeutically editable common variant for ...,T/C,0.10,0.10,"Heterozygous (TC) 13% (329/2504), Homozygous r...","Allele frequency = 0.372163, Heterozygous (TC)...","Allele frequency = 0.00298211, Heterozygous (T...","Allele frequency = 0.0288184, Heterozygous (TC...","Allele frequency = 0, Heterozygous (TC) 0% (0/...","Allele frequency = 0, Heterozygous (TC) 0% (0/...",excision,No,GTGAAATCACTGACTTGGGAACATA_T_AGATTAACTCTTAGTTGC...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135544,chr5,61592482,61592483,"rs253185 G/T, MAF=0.16",A therapeutically editable common variant for ...,G/T,0.84,0.16,"Heterozygous (GT) 18% (457/2504), Homozygous r...","Allele frequency = 0.997731, Heterozygous (GT)...","Allele frequency = 0.981113, Heterozygous (GT)...","Allele frequency = 0.818444, Heterozygous (GT)...","Allele frequency = 0.501984, Heterozygous (GT)...","Allele frequency = 0.861963, Heterozygous (GT)...",excision,Yes,GGTCCCTCTCCTGGCTATCTTGTTT_G_GCTGTAATTATCTCCCAT...
135545,chr5,61592843,61592844,"rs16892657 A/G, MAF=0.1",A therapeutically editable common variant for ...,A/G,0.10,0.10,"Heterozygous (AG) 13% (347/2504), Homozygous r...","Allele frequency = 0.352496, Heterozygous (AG)...","Allele frequency = 0.00198807, Heterozygous (A...","Allele frequency = 0.0244957, Heterozygous (AG...","Allele frequency = 0, Heterozygous (AG) 0% (0/...","Allele frequency = 0.0245399, Heterozygous (AG...",excision,Yes,CTGAAAAACTGTCAGGGGGCACTTC_A_TGTTGGTGCCGTGGTCAA...
135546,chr5,61593465,61593466,"rs10939909 A/G, MAF=0.1",A therapeutically editable co

In [25]:
# add gene as a column
parts = bed['Description'].str.split('A therapeutically editable common variant for the D&D gene ', expand=True)
bed['gene_targeted'] = parts[1]
bed

,chrom,chromStart,chromEnd,rsID,Description,Ref/Alt allele,allele_frequency,minor_allele_frequency,Thousand_genomes_genotypes_unrelated_inds,AFR_genotypes_unrelated_inds,EUR_genotypes_unrelated_inds,AMR_genotypes_unrelated_inds,EAS_genotypes_unrelated_inds,SAS_genotypes_unrelated_inds,CRISPR_editing_strategy_targetable_by,SpCas9 Targetable,25 bp (+/-) flanking the variant,gene_targeted
0,chr16,70251997,70251998,"rs4985407 A/G, MAF=0.46",A therapeutically editable common variant for ...,A/G,0.54,0.46,"Heterozygous (AG) 41% (1046/2504), Homozygous ...","Allele frequency = 0.839637, Heterozygous (AG)...","Allele frequency = 0.538767, Heterozygous (AG)...","Allele frequency = 0.443804, Heterozygous (AG)...","Allele frequency = 0.425595, Heterozygous (AG)...","Allele frequency = 0.344581, Heterozygous (AG)...",excision,Yes,ACGCGCGGCTGCGCATGGGCTGTCG_A_GAGAGCGGCGGGAGGCAC...,AARS1
1,chr16,70253273,70253274,"rs4081753 A/G, MAF=0.12",A therapeutically editable common variant for ...,A/G,0.88,0.12,"Heterozygous (AG) 19% (486/2504), Homozygous r...","Allele frequency = 0.98941, Heterozygous (AG) ...","Allele frequency = 0.797217, Heterozygous (AG)...","Allele frequency = 0.779539, Heterozygous (AG)...","Allele frequency = 0.96627, Heterozygous (AG) ...","Allele frequency = 0.804703, Heterozygous (AG)...","excision,splice site disruption",Yes,GGTGCGAGGTGGTGCTGACCTGGGG_A_ACTTGACACAGGCACGTG...,AARS1
2,chr16,70254828,70254829,"rs7205918 T/C, MAF=0.11",A therapeutically editable common variant for ...,T/C,0.11,0.11,"Heterozygous (TC) 13% (334/2504), Homozygous r...","Allele frequency = 0.381241, Heterozygous (TC)...","Allele frequency = 0.00397614, Heterozygous (T...","Allele frequency = 0.0317003, Heterozygous (TC...","Allele frequency = 0, Heterozygous (TC) 0% (0/...","Allele frequency = 0, Heterozygous (TC) 0% (0/...",excision,Yes,GCTGCGGAAGCCCCGGCGGTAGGCC_T_TGCAGCAACATACCCCAA...,AARS1
3,chr16,70258840,70258841,"rs7186104 T/C, MAF=0.12",A therapeutically editable common variant for ...,T/C,0.88,0.12,"Heterozygous (TC) 19% (487/2504), Homozygous r...","Allele frequency = 0.98941, Heterozygous (TC) ...","Allele frequency = 0.797217, Heterozygous (TC)...","Allele frequency = 0.779539, Heterozygous (TC)...","Allele frequency = 0.96627, Heterozygous (TC) ...","Allele frequency = 0.803681, Heterozygous (TC)...",excision,Yes,aagtgctgggattacaggcgtgagc_t_accgtgcccagccTGCTT...,AARS1
4,chr16,70260570,70260571,"rs74024183 T/C, MAF=0.1",A therapeutically editable common variant for ...,T/C,0.10,0.10,"Heterozygous (TC) 13% (329/2504), Homozygous r...","Allele frequency = 0.372163, Heterozygous (TC)...","Allele frequency = 0.00298211, Heterozygous (T...","Allele frequency = 0.0288184, Heterozygous (TC...","Allele frequency = 0, Heterozygous (TC) 0% (0/...","Allele frequency = 0, Heterozygous (TC) 0% (0/...",excision,No,GTGAAATCACTGACTTGGGAACATA_T_AGATTAACTCTTAGTTGC...,AARS1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135544,chr5,61592482,61592483,"rs253185 G/T, MAF=0.16",A therapeutically editable common variant for ...,G/T,0.84,0.16,"Heterozygous (GT) 18% (457/2504), Homozygous r...","Allele frequency = 0.997731, Heterozygous (GT)...","Allele frequency = 0.981113, Heterozygous (GT)...","Allele frequency = 0.818444, Heterozygous (GT)...","Allele frequency = 0.501984, Heterozygous (GT)...","Allele frequency = 0.861963, Heterozygous (GT)...",excision,Yes,GGTCCCTCTCCTGGCTATCTTGTTT_G_GCTGTAATTATCTCCCAT...,ZSWIM6
135545,chr5,61592843,61592844,"rs16892657 A/G, MAF=0.1",A therapeutically editable common variant for ...,A/G,0.10,0.10,"Heterozygous (AG) 13% (347/2504), Homozygous r...","Allele frequency = 0.352496, Heterozygous (AG)...","Allele frequency = 0.00198807, Heterozygous (A...","Allele frequency = 0.0244957, Heterozygous (AG...","Allele frequency = 0, Heterozygous (AG) 0% (0/...","Allele frequency = 0.0245399, Heterozygous (AG...",excision,Yes,CTGAAAAACTGTCAGGGGGCACTTC_A_TGTTGGTGCCGTGGTCAA...,ZSWIM6
135546,chr5,61593465,61

In [26]:
# remove description column and save
bed.drop(labels=['Description'], axis=1, inplace=True)
bed

,chrom,chromStart,chromEnd,rsID,Ref/Alt allele,allele_frequency,minor_allele_frequency,Thousand_genomes_genotypes_unrelated_inds,AFR_genotypes_unrelated_inds,EUR_genotypes_unrelated_inds,AMR_genotypes_unrelated_inds,EAS_genotypes_unrelated_inds,SAS_genotypes_unrelated_inds,CRISPR_editing_strategy_targetable_by,SpCas9 Targetable,25 bp (+/-) flanking the variant,gene_targeted
0,chr16,70251997,70251998,"rs4985407 A/G, MAF=0.46",A/G,0.54,0.46,"Heterozygous (AG) 41% (1046/2504), Homozygous ...","Allele frequency = 0.839637, Heterozygous (AG)...","Allele frequency = 0.538767, Heterozygous (AG)...","Allele frequency = 0.443804, Heterozygous (AG)...","Allele frequency = 0.425595, Heterozygous (AG)...","Allele frequency = 0.344581, Heterozygous (AG)...",excision,Yes,ACGCGCGGCTGCGCATGGGCTGTCG_A_GAGAGCGGCGGGAGGCAC...,AARS1
1,chr16,70253273,70253274,"rs4081753 A/G, MAF=0.12",A/G,0.88,0.12,"Heterozygous (AG) 19% (486/2504), Homozygous r...","Allele frequency = 0.98941, Heterozygous (AG) ...","Allele frequency = 0.797217, Heterozygous (AG)...","Allele frequency = 0.779539, Heterozygous (AG)...","Allele frequency = 0.96627, Heterozygous (AG) ...","Allele frequency = 0.804703, Heterozygous (AG)...","excision,splice site disruption",Yes,GGTGCGAGGTGGTGCTGACCTGGGG_A_ACTTGACACAGGCACGTG...,AARS1
2,chr16,70254828,70254829,"rs7205918 T/C, MAF=0.11",T/C,0.11,0.11,"Heterozygous (TC) 13% (334/2504), Homozygous r...","Allele frequency = 0.381241, Heterozygous (TC)...","Allele frequency = 0.00397614, Heterozygous (T...","Allele frequency = 0.0317003, Heterozygous (TC...","Allele frequency = 0, Heterozygous (TC) 0% (0/...","Allele frequency = 0, Heterozygous (TC) 0% (0/...",excision,Yes,GCTGCGGAAGCCCCGGCGGTAGGCC_T_TGCAGCAACATACCCCAA...,AARS1
3,chr16,70258840,70258841,"rs7186104 T/C, MAF=0.12",T/C,0.88,0.12,"Heterozygous (TC) 19% (487/2504), Homozygous r...","Allele frequency = 0.98941, Heterozygous (TC) ...","Allele frequency = 0.797217, Heterozygous (TC)...","Allele frequency = 0.779539, Heterozygous (TC)...","Allele frequency = 0.96627, Heterozygous (TC) ...","Allele frequency = 0.803681, Heterozygous (TC)...",excision,Yes,aagtgctgggattacaggcgtgagc_t_accgtgcccagccTGCTT...,AARS1
4,chr16,70260570,70260571,"rs74024183 T/C, MAF=0.1",T/C,0.10,0.10,"Heterozygous (TC) 13% (329/2504), Homozygous r...","Allele frequency = 0.372163, Heterozygous (TC)...","Allele frequency = 0.00298211, Heterozygous (T...","Allele frequency = 0.0288184, Heterozygous (TC...","Allele frequency = 0, Heterozygous (TC) 0% (0/...","Allele frequency = 0, Heterozygous (TC) 0% (0/...",excision,No,GTGAAATCACTGACTTGGGAACATA_T_AGATTAACTCTTAGTTGC...,AARS1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135544,chr5,61592482,61592483,"rs253185 G/T, MAF=0.16",G/T,0.84,0.16,"Heterozygous (GT) 18% (457/2504), Homozygous r...","Allele frequency = 0.997731, Heterozygous (GT)...","Allele frequency = 0.981113, Heterozygous (GT)...","Allele frequency = 0.818444, Heterozygous (GT)...","Allele frequency = 0.501984, Heterozygous (GT)...","Allele frequency = 0.861963, Heterozygous (GT)...",excision,Yes,GGTCCCTCTCCTGGCTATCTTGTTT_G_GCTGTAATTATCTCCCAT...,ZSWIM6
135545,chr5,61592843,61592844,"rs16892657 A/G, MAF=0.1",A/G,0.10,0.10,"Heterozygous (AG) 13% (347/2504), Homozygous r...","Allele frequency = 0.352496, Heterozygous (AG)...","Allele frequency = 0.00198807, Heterozygous (A...","Allele frequency = 0.0244957, Heterozygous (AG...","Allele frequency = 0, Heterozygous (AG) 0% (0/...","Allele frequency = 0.0245399, Heterozygous (AG...",excision,Yes,CTGAAAAACTGTCAGGGGGCACTTC_A_TGTTGGTGCCGTGGTCAA...,ZSWIM6
135546,chr5,61593465,61593466,"rs10939909 A/G, MAF=0.1",A/G,0.10,0.10,"Heterozygous (AG) 13% (339/2504), Homozygous r...","Allele frequency = 0.352496, Heterozygous (AG)...","Allele frequency = 0.00198807, Heterozygous (A...","Allele frequency = 0.0216138, Heterozygous (AG...","Allele frequency = 0, Heterozygous (AG) 0% (0/...","Allele frequency = 0.0184049, Heterozygous (AG...",excisio

In [27]:
bed.columns

Index(['chrom', 'chromStart', 'chromEnd', 'rsID', 'Ref/Alt allele',
       'allele_frequency', 'minor_allele_frequency',
       'Thousand_genomes_genotypes_unrelated_inds',
       'AFR_genotypes_unrelated_inds', 'EUR_genotypes_unrelated_inds',
       'AMR_genotypes_unrelated_inds', 'EAS_genotypes_unrelated_inds',
       'SAS_genotypes_unrelated_inds', 'CRISPR_editing_strategy_targetable_by',
       'SpCas9 Targetable', '25 bp (+/-) flanking the variant',
       'gene_targeted'],
      dtype='object')

In [28]:
# reorder
front = ['gene_targeted']
bed = bed[front + [c for c in bed.columns if c not in front]]

In [29]:
bed

,gene_targeted,chrom,chromStart,chromEnd,rsID,Ref/Alt allele,allele_frequency,minor_allele_frequency,Thousand_genomes_genotypes_unrelated_inds,AFR_genotypes_unrelated_inds,EUR_genotypes_unrelated_inds,AMR_genotypes_unrelated_inds,EAS_genotypes_unrelated_inds,SAS_genotypes_unrelated_inds,CRISPR_editing_strategy_targetable_by,SpCas9 Targetable,25 bp (+/-) flanking the variant
0,AARS1,chr16,70251997,70251998,"rs4985407 A/G, MAF=0.46",A/G,0.54,0.46,"Heterozygous (AG) 41% (1046/2504), Homozygous ...","Allele frequency = 0.839637, Heterozygous (AG)...","Allele frequency = 0.538767, Heterozygous (AG)...","Allele frequency = 0.443804, Heterozygous (AG)...","Allele frequency = 0.425595, Heterozygous (AG)...","Allele frequency = 0.344581, Heterozygous (AG)...",excision,Yes,ACGCGCGGCTGCGCATGGGCTGTCG_A_GAGAGCGGCGGGAGGCAC...
1,AARS1,chr16,70253273,70253274,"rs4081753 A/G, MAF=0.12",A/G,0.88,0.12,"Heterozygous (AG) 19% (486/2504), Homozygous r...","Allele frequency = 0.98941, Heterozygous (AG) ...","Allele frequency = 0.797217, Heterozygous (AG)...","Allele frequency = 0.779539, Heterozygous (AG)...","Allele frequency = 0.96627, Heterozygous (AG) ...","Allele frequency = 0.804703, Heterozygous (AG)...","excision,splice site disruption",Yes,GGTGCGAGGTGGTGCTGACCTGGGG_A_ACTTGACACAGGCACGTG...
2,AARS1,chr16,70254828,70254829,"rs7205918 T/C, MAF=0.11",T/C,0.11,0.11,"Heterozygous (TC) 13% (334/2504), Homozygous r...","Allele frequency = 0.381241, Heterozygous (TC)...","Allele frequency = 0.00397614, Heterozygous (T...","Allele frequency = 0.0317003, Heterozygous (TC...","Allele frequency = 0, Heterozygous (TC) 0% (0/...","Allele frequency = 0, Heterozygous (TC) 0% (0/...",excision,Yes,GCTGCGGAAGCCCCGGCGGTAGGCC_T_TGCAGCAACATACCCCAA...
3,AARS1,chr16,70258840,70258841,"rs7186104 T/C, MAF=0.12",T/C,0.88,0.12,"Heterozygous (TC) 19% (487/2504), Homozygous r...","Allele frequency = 0.98941, Heterozygous (TC) ...","Allele frequency = 0.797217, Heterozygous (TC)...","Allele frequency = 0.779539, Heterozygous (TC)...","Allele frequency = 0.96627, Heterozygous (TC) ...","Allele frequency = 0.803681, Heterozygous (TC)...",excision,Yes,aagtgctgggattacaggcgtgagc_t_accgtgcccagccTGCTT...
4,AARS1,chr16,70260570,70260571,"rs74024183 T/C, MAF=0.1",T/C,0.10,0.10,"Heterozygous (TC) 13% (329/2504), Homozygous r...","Allele frequency = 0.372163, Heterozygous (TC)...","Allele frequency = 0.00298211, Heterozygous (T...","Allele frequency = 0.0288184, Heterozygous (TC...","Allele frequency = 0, Heterozygous (TC) 0% (0/...","Allele frequency = 0, Heterozygous (TC) 0% (0/...",excision,No,GTGAAATCACTGACTTGGGAACATA_T_AGATTAACTCTTAGTTGC...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135544,ZSWIM6,chr5,61592482,61592483,"rs253185 G/T, MAF=0.16",G/T,0.84,0.16,"Heterozygous (GT) 18% (457/2504), Homozygous r...","Allele frequency = 0.997731, Heterozygous (GT)...","Allele frequency = 0.981113, Heterozygous (GT)...","Allele frequency = 0.818444, Heterozygous (GT)...","Allele frequency = 0.501984, Heterozygous (GT)...","Allele frequency = 0.861963, Heterozygous (GT)...",excision,Yes,GGTCCCTCTCCTGGCTATCTTGTTT_G_GCTGTAATTATCTCCCAT...
135545,ZSWIM6,chr5,61592843,61592844,"rs16892657 A/G, MAF=0.1",A/G,0.10,0.10,"Heterozygous (AG) 13% (347/2504), Homozygous r...","Allele frequency = 0.352496, Heterozygous (AG)...","Allele frequency = 0.00198807, Heterozygous (A...","Allele frequency = 0.0244957, Heterozygous (AG...","Allele frequency = 0, Heterozygous (AG) 0% (0/...","Allele frequency = 0.0245399, Heterozygous (AG...",excision,Yes,CTGAAAAACTGTCAGGGGGCACTTC_A_TGTTGGTGCCGTGGTCAA...
135546,ZSWIM6,chr5,61593465,61593466,"rs10939909 A/G, MAF=0.1",A/G,0.10,0.10,"Heterozygous (AG) 13% (339/2504), Homozygous r...","Allele frequency = 0.352496, Heterozygous (AG)...","Allele frequency = 0.00198807, Heterozygous (A...","Allele frequency = 0.0216138, Heterozygous (AG...","Allele frequency = 0, Heterozygous (AG) 0% (0/...","Allele frequency = 0.0184049, Heterozygous (AG...",

In [30]:
# save
bed.to_csv('/wynton/home/capra/gramey02/dnd_project/results/RUN_MULTIALLELIC/summary_files/common_var_bed/all_dnd_candidate_vars.bed')